# Notebook 2: Limpieza, Transformación e Integración de Fuentes

**DataJam Edición 4 — Universidad Distrital Francisco José de Caldas**

## Objetivo
Limpiar cada dataset, normalizar las llaves geográficas (localidad, UPL) y construir las tablas integradas que alimentan el análisis.

## Pasos
1. Normalización de nombres de localidades
2. Mapeo UPL → Localidad
3. Integración: Pobreza + Deserción + Matrícula
4. Procesamiento de la Encuesta Multipropósito
5. Exportación de tablas limpias

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath('.')).resolve()
if '__vsc_ipynb_file__' in dir():
    PROJECT_ROOT = Path(__vsc_ipynb_file__).resolve().parent.parent
else:
    for _ in range(10):
        if (PROJECT_ROOT / 'requirements.txt').exists() and (PROJECT_ROOT / 'scripts').exists():
            break
        PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Datos en: {DATA_DIR}")
print(f"Salida en: {OUTPUT_DIR}")

Datos en: /home/aletarget/Documents/DataJam/ProblemasEconomicosYRelacionconperformanceenestudiantes/DataJam/data
Salida en: /home/aletarget/Documents/DataJam/ProblemasEconomicosYRelacionconperformanceenestudiantes/DataJam/output


## 2.1 Diccionarios de mapeo territorial

Bogotá tiene 20 localidades y 33 UPLs (Unidades de Planeamiento Local). Necesitamos un mapeo consistente.

In [2]:
# Códigos de localidad
LOC_NOMBRES = {
    '01': 'Usaquén', '02': 'Chapinero', '03': 'Santa Fe', '04': 'San Cristóbal',
    '05': 'Usme', '06': 'Tunjuelito', '07': 'Bosa', '08': 'Kennedy',
    '09': 'Fontibón', '10': 'Engativá', '11': 'Suba', '12': 'Barrios Unidos',
    '13': 'Teusaquillo', '14': 'Los Mártires', '15': 'Antonio Nariño',
    '16': 'Puente Aranda', '17': 'La Candelaria', '18': 'Rafael Uribe Uribe',
    '19': 'Ciudad Bolívar', '20': 'Sumapaz'
}

# Mapeo UPL → Código de localidad
UPL_LOCALIDAD = {
    'UPL01': '20', 'UPL02': '19', 'UPL03': '19', 'UPL04': '05', 'UPL05': '04',
    'UPL06': '03', 'UPL07': '01', 'UPL08': '08', 'UPL09': '09', 'UPL10': '11',
    'UPL11': '10', 'UPL12': '11', 'UPL13': '08', 'UPL14': '16', 'UPL15': '12',
    'UPL16': '02', 'UPL17': '01', 'UPL18': '11', 'UPL19': '10', 'UPL20': '18',
    'UPL21': '06', 'UPL22': '03', 'UPL23': '04', 'UPL24': '05', 'UPL25': '19',
    'UPL26': '07', 'UPL27': '08', 'UPL28': '08', 'UPL29': '09', 'UPL30': '07',
    'UPL31': '16', 'UPL32': '13', 'UPL33': '12',
}

# Mapeo inverso: nombre de localidad → código
NOMBRE_A_COD = {
    'Usaquén': '01', 'Chapinero': '02', 'Santa Fe': '03', 'San Cristóbal': '04',
    'Usme': '05', 'Tunjuelito': '06', 'Bosa': '07', 'Kennedy': '08',
    'Fontibón': '09', 'Engativá': '10', 'Suba': '11', 'Barrios Unidos': '12',
    'Teusaquillo': '13', 'Los Mártires': '14', 'Antonio Nariño': '15',
    'Puente Aranda': '16', 'La Candelaria': '17', 'Rafael Uribe Uribe': '18',
    'Ciudad Bolívar': '19', 'Sumapaz': '20',
}

print(f"Localidades: {len(LOC_NOMBRES)}")
print(f"UPLs mapeadas: {len(UPL_LOCALIDAD)}")

Localidades: 20
UPLs mapeadas: 33


## 2.2 Carga y limpieza: Pobreza

In [3]:
df_pob = pd.read_csv(DATA_DIR / 'pobreza' / 'osb_demografia-pobrezaygini.csv',
                     sep=';', encoding='latin-1')
df_pob.columns = ['Año', 'Localidad', 'Indicador', 'Categoría', 'Sexo', 'Valor']
df_pob['Valor'] = df_pob['Valor'].astype(str).str.replace(',', '.').astype(float)
df_pob['Año'] = df_pob['Año'].astype(int)

# Agregar código de localidad
df_pob['COD_LOCA'] = df_pob['Localidad'].map(NOMBRE_A_COD)

print(f"Registros: {len(df_pob)}")
print(f"Nulos en Valor: {df_pob['Valor'].isna().sum()}")
print(f"Localidades sin código: {df_pob[df_pob['COD_LOCA'].isna()]['Localidad'].unique()}")

Registros: 447
Nulos en Valor: 0
Localidades sin código: ['Bogotá D.C.']


## 2.3 Carga y limpieza: Deserción por UPL

In [4]:
with open(DATA_DIR / 'desercion_upl' / 'tasas_upl.geojson') as f:
    gj = json.load(f)

deser_rows = []
for feat in gj['features']:
    p = feat['properties']
    cod_upl = p.get('CODIGO_UPL') or p.get('codigo_upl', '')
    nom_upl = p.get('NOM_UPL') or p.get('nom_upl', '')
    deser_rows.append({
        'Cod_UPL': cod_upl,
        'NOM_UPL': nom_upl,
        'Desercion_Of': p.get('TtotalDeserOf_UPL') or p.get('ttotal_deser_of_upl', 0),
        'Reprobacion_Of': p.get('TtotalReprOf_UPL') or p.get('ttotal_repr_of_upl', 0),
        'Aprobacion_Of': p.get('TtotalAprOf_UPL') or p.get('ttotal_apr_of_upl', 0),
        'Desercion_NOf': p.get('TtotalDeserNOf_UPL') or p.get('ttotal_deser_n_of_upl', 0),
    })

df_deser = pd.DataFrame(deser_rows)
df_deser['COD_LOCA'] = df_deser['Cod_UPL'].map(UPL_LOCALIDAD)
df_deser['Localidad'] = df_deser['COD_LOCA'].map(LOC_NOMBRES)

print(f"UPLs: {len(df_deser)}")
print(f"\nDeserción oficial promedio: {df_deser['Desercion_Of'].mean():.2f}%")
print(f"Reprobación oficial promedio: {df_deser['Reprobacion_Of'].mean():.2f}%")
df_deser.sort_values('Desercion_Of', ascending=False).head(10)

UPLs: 33

Deserción oficial promedio: 2.72%
Reprobación oficial promedio: 7.71%


,Cod_UPL,NOM_UPL,Desercion_Of,Reprobacion_Of,Aprobacion_Of,Desercion_NOf,COD_LOCA,Localidad
28,UPL24,24,4.681873,10.804322,84.513806,1.314741,05,Usme
5,UPL25,25,4.030922,13.583655,82.385422,1.476015,19,Ciudad Bolívar
4,UPL32,32,4.013761,12.844037,83.142202,1.921230,13,Teusaquillo
8,UPL33,33,3.987953,8.318621,87.693426,0.861563,12,Barrios Unidos
29,UPL23,23,3.647046,7.227580,89.125374,2.160864,04,San Cristóbal
6,UPL27,27,3.597516,7.304602,89.097882,4.539943,08,Kennedy
24,UPL08,08,3.365792,8.488569,88.145639,1.605996,08,Kennedy
13,UPL26,26,3.133529,7.017636,89.848835,1.293998,07,Bosa
27,UPL04,04,3.101675,6.644077,90.254247,1.465798,05,Usme
7,UPL18,18,3.069754,5.188317,91.741929,1.077420,11,Suba


## 2.4 Integración: Pobreza × Deserción × Matrícula a nivel de localidad

In [5]:
# Pobreza monetaria 2021 por localidad
pobreza_loc = df_pob[
    (df_pob['Indicador'] == 'Pobreza monetaria') &
    (df_pob['Año'] == 2021) &
    (df_pob['Sexo'].str.contains('Ambos', na=False)) &
    (~df_pob['Localidad'].str.contains('Bogot', na=False))
][['Localidad', 'Valor', 'COD_LOCA']].rename(columns={'Valor': 'Pobreza_Monetaria'})

# Agregar deserción y reprobación (promedio por localidad)
deser_loc = df_deser.groupby('COD_LOCA').agg(
    Desercion_Of=('Desercion_Of', 'mean'),
    Reprobacion_Of=('Reprobacion_Of', 'mean'),
).reset_index()

# Matrícula
mat_path = DATA_DIR / 'matricula' / 'matriculaciones.geojson'
if mat_path.exists():
    with open(mat_path) as f:
        gj_mat = json.load(f)
    mat_rows = []
    for feat in gj_mat['features']:
        p = feat['properties']
        cod_loca = p.get('COD_LOCA') or p.get('cod_loca') or p.get('loc', '')
        matricula = p.get('TMATRIC_GE') or p.get('tmatric_ge', 0)
        mat_rows.append({
            'COD_LOCA': str(cod_loca).zfill(2) if cod_loca else '',
            'Matricula': matricula or 0,
        })
    df_mat = pd.DataFrame(mat_rows)
    mat_loc = df_mat.groupby('COD_LOCA').agg(
        Matricula=('Matricula', 'sum'),
        Sedes=('Matricula', 'count'),
    ).reset_index()
    mat_loc['Est_por_Sede'] = mat_loc['Matricula'] / mat_loc['Sedes']
else:
    mat_loc = pd.DataFrame(columns=['COD_LOCA', 'Matricula', 'Sedes', 'Est_por_Sede'])

# Integrar
integrado = pobreza_loc.merge(deser_loc, on='COD_LOCA', how='left')
integrado = integrado.merge(mat_loc, on='COD_LOCA', how='left')
integrado = integrado.dropna(subset=['Desercion_Of'])

print(f"Tabla integrada: {len(integrado)} localidades")
integrado.sort_values('Pobreza_Monetaria', ascending=False)

Tabla integrada: 17 localidades


,Localidad,Pobreza_Monetaria,COD_LOCA,Desercion_Of,Reprobacion_Of,Matricula,Sedes,Est_por_Sede
4,Usme,57.81,05,3.891774,8.724200,NaN,NaN,NaN
18,Ciudad Bolívar,57.37,19,2.729831,8.977344,NaN,NaN,NaN
6,Bosa,53.18,07,2.351789,7.162200,NaN,NaN,NaN
17,Rafael Uribe Uribe,49.95,18,2.268404,7.442632,NaN,NaN,NaN
3,San Cristóbal,48.45,04,2.696818,7.699415,NaN,NaN,NaN
2,Santa Fe,47.69,03,2.754387,7.435757,NaN,NaN,NaN
5,Tunjuelito,40.19,06,2.258850,8.530093,NaN,NaN,NaN
19,Sumapaz,38.67,20,3.004292,7.725322,NaN,NaN,NaN
7,Kennedy,37.02,08,3.011505,7.307488,NaN,NaN,NaN
10,Suba,25.82,11,2.877691,7.388676,NaN,NaN,NaN


## 2.5 Exportar tablas limpias para análisis

In [6]:
# Guardar tabla integrada para los siguientes notebooks
integrado.to_csv(OUTPUT_DIR / 'tabla_integrada_localidad.csv', index=False)
df_deser.to_csv(OUTPUT_DIR / 'desercion_por_upl.csv', index=False)

print("✓ Tablas exportadas:")
print(f"  - output/tabla_integrada_localidad.csv ({len(integrado)} filas)")
print(f"  - output/desercion_por_upl.csv ({len(df_deser)} filas)")
print("\n→ Siguiente: Notebook 03 (Análisis Exploratorio y Correlaciones)")

✓ Tablas exportadas:
  - output/tabla_integrada_localidad.csv (17 filas)
  - output/desercion_por_upl.csv (33 filas)

→ Siguiente: Notebook 03 (Análisis Exploratorio y Correlaciones)
